Group 3 Study Planner based on Career

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries for our entire adventure ---
import os
import re
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.tools import google_search, ToolContext
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from google.adk.tools.agent_tool import AgentTool
from getpass import getpass

print("✅ All libraries are ready to go!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [3]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "study_planner_gp3"

In [4]:
# --- Agent Definitions for a Sequential Workflow ---

# Using a reliable model ID for compatibility
MODEL_ID = "gemini-2.5-flash"

# Specialist Agent 1
course_finder_agent = Agent(
    name="course_finder_agent", model=MODEL_ID, tools=[google_search],
    instruction="You are a Specialized Educational Resource Agent. Find high-quality courses and documentation for the user's request.",
    output_key="course_result"
)

# Specialist Agent 2
resources_finder_agent = Agent(
    name="resources_finder_agent", model=MODEL_ID, tools=[google_search],
    instruction="You are a Multimedia Resource Specialist. Find YouTube tutorials and blogs to supplement the courses found in {course_result}.",
    output_key="resources_result"
)

# Specialist Agent 3
weekly_planner_agent = Agent(
    name="weekly_planner_agent", model=MODEL_ID, tools=[google_search],
    instruction="You are a Productivity Coach. Organize the courses and resources into a realistic weekly schedule.",
    output_key="weekly_planner_result"
)

coach_agent = SequentialAgent(
    name="coach_agent",
    sub_agents=[course_finder_agent, resources_finder_agent, weekly_planner_agent],
    description="A tool that identifies courses, curates resources, and creates a weekly schedule."
)

concierge_agent = Agent(
    name="concierge_agent",
    model=MODEL_ID,
    instruction="""You are a Strategic Academic Coordinator.

    Your process:
    1. Call the 'coach_agent' tool with the user's request to get a full curriculum and schedule.
    2. Once you receive the tool's output, synthesize it into a final response.

    Output Format:
    - 'Strategic Overview': Summarize the theme.
    - 'Resource-to-Course Mapping': Connect resources to specific objectives from the tool output You are a Multimedia Resource Specialist. Find YouTube tutorials and blogs to supplement the courses found in {course_result}.
    - 'Weekly Focus': Top 3 priorities.
    """,
    tools=[AgentTool(agent=coach_agent)]
)

print("✅ Agents updated with correct context handling and model IDs.")

✅ Agents updated with correct context handling and model IDs.


In [ ]:
# The master dictionary of all our executable agents and workflows
worker_agents = {
    "course_finder_agent": course_finder_agent,
    "resources_finder_agent": resources_finder_agent,
    "weekly_planner_agent": weekly_planner_agent,
    "coach_agent": coach_agent,
    "concierge_agent": concierge_agent,
}

In [ ]:
import gradio as gr
import asyncio

def run_agent_ui(career, level, time_limit):
    # This function wraps the async agent call for Gradio
    async def process():
        query = f"I am at a {level} level and want to become a {career}. My time limitation is {time_limit}. Please generate a study plan."

        # Create a new session for this request
        study_session = await session_service.create_session(
            app_name=concierge_agent.name,
            user_id=my_user_id
        )

        # Call the agent using the existing helper function
        # We set is_router=True to avoid printing to the console and just get the text
        response = await run_agent_query(concierge_agent, query, study_session, my_user_id, is_router=True)
        return response

    # Run the async process in the current event loop
    return asyncio.run(process())

career_options = [
    "Software Developer / Engineer",
    "AI / Machine Learning Engineer",
    "Data Scientist",
    "Cybersecurity Analyst / Engineer",
    "Cloud Architect",
    "DevOps Engineer",
    "Data Engineer",
    "Web Developer",
    "Computer Systems Analyst",
    "Software Architect"
]

level_options = ["Beginner", "Intermediate", "Advanced"]

demo = gr.Interface(
    fn=run_agent_ui,
    inputs=[
        gr.Dropdown(choices=career_options, label="Target Career Path"),
        gr.Dropdown(choices=level_options, label="Current Level"),
        gr.Textbox(label="Time Limitation", placeholder="e.g. 6 months, 10 hours/week")
    ],
    outputs=gr.Markdown(label="Your Strategic Roadmap"),
    title="Group 3 AI Study Planner",
    description="Fill in your details below to generate a personalized study plan using our multi-agent system."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://247acd6300fb71bd57.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🚀 Running query for agent: 'concierge_agent' in session: '848372aa-b4dc-42d1-a842-4ea83e9c308d'...


Created dataset file at: .gradio/flagged/dataset1.csv

🚀 Running query for agent: 'concierge_agent' in session: '2be996d2-a7ca-47c3-8563-ea9413b64b8f'...

🚀 Running query for agent: 'concierge_agent' in session: '223124b0-9986-4b60-b85e-a793dd98b769'...

🚀 Running query for agent: 'concierge_agent' in session: '344d7a5e-1c8e-4b03-8549-4bc88799e0e7'...

🚀 Running query for agent: 'concierge_agent' in session: '75a468f2-2104-4157-8a7a-9ccc7e9d9aba'...

🚀 Running query for agent: 'concierge_agent' in session: '021cb348-3724-47ec-abba-5ca013b86102'...

🚀 Running query for agent: 'concierge_agent' in session: '96d0d0ba-441d-4873-ba4d-3f63f5f911c1'...
